# 01 — Pose Extraction on Google Colab (YOLO11-Pose HD + ByteTrack)

**Project:** CVF25PRJ10 • CVFP26BCPPRJC012  
**Target:** High-Accuracy Pose Extraction with GPU Acceleration (NVIDIA T4 / A100 / V100).  
**Models:** YOLO11m-Pose (`yolo11m-pose.pt`, 1280px HD) + ByteTrack tracker.  

Sổ tay này giúp trích xuất khung xương 17 điểm COCO cho toàn bộ các session (`session_001` đến `session_006`) hoặc video bất kỳ trên Google Colab với tốc độ siêu nhanh (~15 - 20 phút cho toàn bộ 79 phút video khi dùng GPU).


## Colab usage rule

Do not duplicate model/data-processing logic in this notebook. Import reusable code from `src/`. Do not commit raw private videos or large checkpoints to Git.


In [ ]:
# Optional when running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# Tự động clone repository nếu đang chạy trên Google Colab
if Path('/content').is_dir() and not (Path('/content/observable-learning-engagement') / 'src').is_dir():
    print('🚀 Phát hiện Google Colab. Đang tự động clone repository...')
    os.system('git clone -b feature/ver2 https://github.com/hquannnguyen/observable-learning-engagement.git /content/observable-learning-engagement')

if (Path('/content/observable-learning-engagement') / 'src').is_dir():
    os.chdir('/content/observable-learning-engagement')
    if '/content/observable-learning-engagement' not in sys.path:
        sys.path.insert(0, '/content/observable-learning-engagement')

def _is_project_root(path: Path) -> bool:
    return (
        (path / 'AGENTS.md').is_file()
        and (path / 'configs' / 'baseline.yaml').is_file()
        and (path / 'scripts' / 'check_setup.py').is_file()
        and (path / 'src').is_dir()
    )

def find_project_root(start: Path | None = None) -> Path:
    env_root = os.environ.get('CVF_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if _is_project_root(candidate):
            return candidate
        raise RuntimeError(f'CVF_PROJECT_ROOT is not a valid project root: {candidate}')

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    # Typical Colab case: cwd=/content and the cloned repo is one direct child.
    child_matches = [
        child for child in start.iterdir()
        if child.is_dir() and _is_project_root(child)
    ] if start.is_dir() else []
    if len(child_matches) == 1:
        return child_matches[0]
    if len(child_matches) > 1:
        raise RuntimeError(
            'Multiple project roots found. Set CVF_PROJECT_ROOT explicitly: ' +
            ', '.join(map(str, child_matches))
        )

    # Automatic fallback for Google Colab environment:
    if Path('/content').is_dir() and (start == Path('/content') or Path('/content') in start.parents):
        repo_candidate = Path('/content/observable-learning-engagement')
        if not _is_project_root(repo_candidate):
            import subprocess
            print('🔍 Đang tự động clone repository vào Google Colab (/content)...')
            subprocess.run(
                ['git', 'clone', '-b', 'feature/ver2', 'https://github.com/hquannnguyen/observable-learning-engagement.git', str(repo_candidate)],
                check=False,
            )
        if _is_project_root(repo_candidate):
            return repo_candidate

    raise RuntimeError(
        'Project root not found. Clone/open the repository, or set CVF_PROJECT_ROOT.'
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository root:', ROOT)


## 1. Kiểm tra phần cứng GPU

Trên menu Colab: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** (hoặc A100 GPU).

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
gpu_ok = torch.cuda.is_available()
print('CUDA Available :', gpu_ok)
if gpu_ok:
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    print('⚠️ Đang chạy trên CPU. Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU để tăng tốc!')


## 2. Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt các dependencies từ requirements.txt
!pip install -q ultralytics==8.3.242 pyarrow==20.0.0 pyyaml==6.0.3


## 3. Kết nối Google Drive (Tùy chọn)

Nếu bạn lưu video `session_001` đến `session_006` trên Google Drive, mount Drive bên dưới để đọc trực tiếp:

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive đã kết nối thành công!')
except Exception as e:
    print('Bỏ qua mount Drive nếu chạy cục bộ hoặc không dùng Drive.')


## 4. Chạy trích xuất Pose bằng YOLO11-Pose HD (1280px) + ByteTrack

Chọn các session cần chạy (ví dụ `session_003`, `session_004`, `session_005`, `session_006`) hoặc toàn bộ các session:

In [ ]:
# Chạy trích xuất các session còn lại (session_003 đến session_006) bằng GPU:
!python scripts/batch_extract_sessions.py \
    --sessions session_003 session_004 session_005 session_006 \
    --config configs/experiments/pose_yolo11m_hd.yaml \
    --imgsz 1280 \
    --device cuda

# Ghi chú: Nếu muốn chạy cả 6 session (session_001 đến session_006):
# !python scripts/batch_extract_sessions.py --config configs/experiments/pose_yolo11m_hd.yaml --imgsz 1280 --device cuda


## 5. Thống kê kết quả & Kiểm tra file đã trích xuất

In [ ]:
import pandas as pd
from pathlib import Path

pose_dir = Path('data/poses')
files = sorted(pose_dir.glob('session_*_cam_A_poses.parquet'))

print(f'Tìm thấy {len(files)} file Parquet:')
for f in files:
    df = pd.read_parquet(f)
    print(f' • {f.name:<32} | {len(df):>7} detections | {len(df["track_id"].unique()):>3} unique tracks')


## 6. Nén và tải kết quả về máy (hoặc lưu sang Google Drive)

In [ ]:
# Nén toàn bộ file poses thành file zip để tải về chỉ với 1 click
!zip -r yolo11_sessions_poses.zip data/poses/session_*_cam_A_poses.*

# Tự động tải về máy nếu dùng trình duyệt
try:
    from google.colab import files
    files.download('yolo11_sessions_poses.zip')
except Exception:
    print('File nén lưu tại yolo11_sessions_poses.zip')
